# 13. Context interventions and identity geometry

![Context interventions](../images/13_context_interventions.svg)

**Learning goals:** create a paired context substitution, measure normalized loss change and pairwise geometry distortion, evaluate identity with enrollment-only centroids and cosine retrieval, and compare hard with soft completion.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 13
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)

def normalize_rows(z, eps=1e-12):
    z = np.asarray(z, dtype=float)
    norms = np.linalg.norm(z, axis=1, keepdims=True)
    if np.any(norms <= eps):
        raise ValueError("cosine geometry requires nonzero row vectors")
    return z / norms

print(f"NumPy {np.__version__}; seed={SEED}")

## 1. Generate paired identity and context observations

Each of five identities has a latent direction in four dimensions. For every identity and repeat, baseline and intervention rows share the same identity and noise, while the intervention adds a context shift. This pairing isolates the intended context change in the simulation. Arrays `Z_base` and `Z_intervention` both have shape `(n_pairs, representation_dim)`.

In [ ]:
n_id, repeats, p = 5, 8, 4
identity_centers = normalize_rows(rng.normal(size=(n_id, p)))
labels = np.repeat(np.arange(n_id), repeats)
paired_noise = rng.normal(0, 0.08, size=(n_id * repeats, p))
context_shift = np.array([0.65, -0.30, 0.15, 0.20])
Z_base = identity_centers[labels] + paired_noise
Z_intervention = identity_centers[labels] + paired_noise + context_shift
assert Z_base.shape == Z_intervention.shape == (40, 4)
assert np.array_equal(labels[:repeats], np.zeros(repeats, dtype=int))
print("paired representations:", Z_base.shape, "context shift norm:", round(np.linalg.norm(context_shift), 3))

## 2. Loss contrasts and geometry matching

We define loss as squared distance to the correct identity center. The symmetric contrast $(L_i-L_b)/((|L_i|+|L_b|)/2+\epsilon)$ has sign symmetry and a comparable scale across pairs. Geometry matching compares upper-triangular pairwise cosine distances. Cosine geometry explicitly rejects zero-norm rows because a collapsed zero vector has no direction. Pairwise matrices require $O(n^2)$ memory, so sample a fixed set of pairs for large datasets.

In [ ]:
def normalized_contrast(base, intervention, eps=1e-12):
    scale = (np.abs(base) + np.abs(intervention)) / 2
    return (intervention - base) / (scale + eps)

def cosine_matrix(z):
    z_normalized = normalize_rows(z)
    return 1 - np.clip(z_normalized @ z_normalized.T, -1, 1)

loss_base = np.sum((Z_base - identity_centers[labels]) ** 2, axis=1)
loss_intervention = np.sum((Z_intervention - identity_centers[labels]) ** 2, axis=1)
contrast = normalized_contrast(loss_base, loss_intervention)
upper = np.triu_indices(len(labels), k=1)
Db, Di = cosine_matrix(Z_base), cosine_matrix(Z_intervention)
geometry_mae = np.mean(np.abs(Db[upper] - Di[upper]))
assert np.allclose(np.diag(Db), 0, atol=1e-12)
collapsed_rejected = False
try:
    cosine_matrix(np.zeros((len(labels), p)))
except ValueError:
    collapsed_rejected = True
assert collapsed_rejected
print(f"mean normalized loss contrast={contrast.mean():.3f}")
print(f"pairwise geometry MAE={geometry_mae:.3f}; zero collapse rejected={collapsed_rejected}")

## 3. Separate enrollment from probes

The first four repeats per identity form enrollment in the baseline context. The remaining four paired rows form probes in the intervention context. No probe is used to estimate centroids. For cosine centroids, normalize each enrollment row before averaging so high-norm rows do not receive extra weight, then normalize each centroid. In real data, split participant, sequence, or source groups rather than nearby rows.

In [ ]:
repeat_index = np.tile(np.arange(repeats), n_id)
enroll_mask = repeat_index < repeats // 2
probe_mask = ~enroll_mask
assert not np.any(enroll_mask & probe_mask)
assert enroll_mask.sum() == probe_mask.sum() == 20

def fit_cosine_centroids(z, y):
    classes = np.unique(y)
    unit_z = normalize_rows(z)
    centroids = np.stack([unit_z[y == k].mean(axis=0) for k in classes])
    return classes, normalize_rows(centroids)

classes, centroids = fit_cosine_centroids(Z_base[enroll_mask], labels[enroll_mask])
probe_z, probe_y = Z_intervention[probe_mask], labels[probe_mask]
pred = classes[np.argmax(normalize_rows(probe_z) @ centroids.T, axis=1)]
centroid_accuracy = np.mean(pred == probe_y)
assert np.allclose(np.linalg.norm(centroids, axis=1), 1.0)
print(f"cross-context nearest-centroid accuracy={centroid_accuracy:.3f}")

## 4. Individual cosine retrieval

Retrieval keeps every enrollment item instead of averaging identities. After one row normalization, matrix multiplication returns all probe-gallery similarities. `np.ascontiguousarray` stores the transposed gallery in a BLAS-friendly memory layout for repeated products. A probe succeeds when its nearest enrollment item has the same identity. This synthetic example has continuous noise, so exact ties are unlikely; Lesson 12 supplies tie-aware metrics when ties occur.

In [ ]:
gallery_z, gallery_y = Z_base[enroll_mask], labels[enroll_mask]
gallery_transpose = np.ascontiguousarray(normalize_rows(gallery_z).T)
similarity = normalize_rows(probe_z) @ gallery_transpose
nearest = np.argmax(similarity, axis=1)
retrieval_accuracy = np.mean(gallery_y[nearest] == probe_y)
assert similarity.shape == (len(probe_z), len(gallery_z))
print(f"cross-context retrieval accuracy={retrieval_accuracy:.3f}")

fig, ax = plt.subplots(figsize=(6, 4))
image = ax.imshow(similarity, aspect="auto", cmap="viridis")
ax.set(xlabel="enrollment item", ylabel="intervention probe", title="Cosine similarity")
fig.colorbar(image, ax=ax, label="similarity")
plt.tight_layout()
plt.show()

## 5. Hard and soft factor completion

A partial query leaves one binary context unknown. Hard completion selects the most probable context. Soft completion keeps both possibilities. Expected loss is generally safer than computing loss at the mean representation because nonlinear losses do not commute with expectation.

In [ ]:
probabilities = np.array([0.60, 0.40])
candidate_representations = np.array([[1.0, 0.0], [-1.0, 0.0]])
target = np.array([1.0, 0.0])
candidate_losses = np.sum((candidate_representations - target) ** 2, axis=1)
hard_index = np.argmax(probabilities)
hard_loss = candidate_losses[hard_index]
expected_loss = probabilities @ candidate_losses
mean_representation = probabilities @ candidate_representations
loss_at_mean = np.sum((mean_representation - target) ** 2)
assert not np.isclose(expected_loss, loss_at_mean)
print(f"hard loss={hard_loss:.2f}; expected loss={expected_loss:.2f}; loss at mean={loss_at_mean:.2f}")

## Exercises, construct validity, and takeaways

1. Set `context_shift` to zero. Predict the loss contrast and geometry error.
2. Fit centroids using all rows, including probes. Why is the resulting accuracy invalid?
3. Increase the rare completion's loss and compare hard with expected loss.

**Brief answers:** a zero shift gives contrast and geometry error near zero. Probe-fitted centroids leak evaluation information. Hard completion remains fixed while expected loss grows with the rare but costly outcome.

**Construct validity:** invariance is meaningful only if identity remains decodable and the intervention truly changes context. A zero-vector collapse is outside the cosine domain and must be rejected, not scored as perfect stability. Add a positive context-sensitive control, a negative control, and group-separated enrollment and probes.

**Takeaway:** loss, pairwise geometry, centroids, and retrieval expose different kinds of context sensitivity. None replaces a valid intervention design.

## Continue learning

[Previous notebook: 12](12_blockwise_distances_and_ranking.ipynb) | [Lecture](../lectures/13_context_interventions.md) | [Curriculum](../README.md) | [Next notebook: 14](14_paired_inference.ipynb)